# Set Up pwd and auto updates

In [ ]:
from pathlib import Path
import os
import subprocess
# Get the top-level directory of the current git repo
PROJECT_ROOT = Path(
    subprocess.check_output(
        ["git", "rev-parse", "--show-toplevel"], text=True
    ).strip()
)

os.chdir(PROJECT_ROOT)


# Enable auto-reloading of custom modules
%load_ext autoreload
%autoreload 2

%pwd
pd.set_option('display.max_columns', None)

In [ ]:

from season.configs import summarize_config
from collect_external_data.expected_counts import get_expected_counts
from collect_external_data.road_geom import get_road_geometry

from season.persons import SeasonPerson
from season.configs import ScheduleSpecs, SeasonConfig, DayParams, make_season_config, PopulationParams
from traffic.model.bus_system_cost import BusCostConfig

from season.season_orchestrator import SeasonOrchestrator
from traffic.model.hybrid_collector import HybridCollectorConfig
from traffic.model.tolling import (
    TollConfig, 
    VolumeSignal, 
    FlowSignal, 
    PiecewiseLinearTransform, 
    StepTransform, 
    PITransform
)

import numpy as np
import pandas as pd
from pathlib import Path

from scipy.stats import norm, lognorm, skewnorm, truncnorm, uniform

import seaborn as sns



# Ensure Data Exists 

In [ ]:
get_road_geometry()
get_expected_counts()


# Defining the Population Perams 

for small runs: 
- PopulationParams.population_size == make_season_config.max_persons & small n 
- 


In [ ]:
config = make_season_config(
    # ── Identity ──
    season_id='',
    run_description='',
    seed=33,

    # ── Simulation bounds ──
    n_days=3,
    max_steps=99999,
    start_hr=7,

    # ── Schedules (day-varying) ──
    traffic_percentile_schedule=ScheduleSpecs(mode='list', value=[85, 85, 85]),
    bus_interval_schedule=ScheduleSpecs(mode='static', value=30),
    crashes_schedule=ScheduleSpecs(mode='static', value=0),
    canyon_closures_schedule=None,

    # ── Population ──
    population_params=PopulationParams(
        population_size=1500,
        prior_car=22.0,
        prior_bus=40.0,
        time_decay_rate=0.1,
        prior_weight=1.0,
        uncertainty_multiplier=1.0,
    ),

    # ── Tolling ──
    toll=TollConfig(
        signal=VolumeSignal(),
        transform=PITransform(target=300, kp=0.5, ki=0.05, toll_min=0, toll_max=50),
        update_every_n_steps=60,
        rounding=0.10,
    ),
    bus_user_fee=0.0,
    bus_capacity=60,
    
    # ── Bus Cost ──
    bus_cost_config=BusCostConfig.default(),
    
    # ── Data collection ──
    collect_every_n=60,
    hybrid_collector_config=HybridCollectorConfig(
        max_steps=100000,
        tier1_enabled=True,
        tier2_enabled=False,
        tier3_enabled=False,
        tier4_enabled=False,
        tier1_interval=60,
        tier2_sample_interval=30,
        tier4_snapshot_interval=500,
        tier1_scalars=[
            'step', 'p_generate', 'current_toll', 'vehicle_count', 'active_cars', 'bus_riders_waiting',
            'active_buses', 'total_finished', 'bus_mode_share_recent',
        ],
        tier1_window_scalars=[
            'recent_travel_time_avg',
            'rolling_count_vehicles_generated',
            'rolling_count_persons_generated',
        ],
        tier1_histograms=['implicit_sl_delta'],
        tier2_max_samples=3000,
        tier2_max_agents_per_sample=150,
        tier4_snapshot_on_crash=True,
        tier4_max_snapshots=20,
        tier1_window_seconds=600,
    ),

)


summarize_config(config, high_only=False)

In [ ]:
orchestrator = SeasonOrchestrator(season_config=config, store_data=True)
orchestrator.run_season()


In [ ]:
# Read the day_0_model_ts parquet produced by the season run
parquet_path = PROJECT_ROOT / "data" / "season_outputs" / "speed_test2" / "day_2_model_ts.parquet"

if not parquet_path.exists():
    raise FileNotFoundError(f"Parquet file not found: {parquet_path}")

df_day = pd.read_parquet(parquet_path)

print(f"Loaded: {parquet_path}")
print("Shape:", df_day.shape)
print("\nColumn dtypes:")
print(df_day.dtypes)

# Show a quick sample
try:
    display(df_day.head(10))
except NameError:
    print(df_day.head(10))

In [ ]:
df_day

In [ ]:
orchestrator.last_model_run

In [ ]:
import cProfile
import pstats

# some stuff used for optimization

def main():
    # Example usage of SeasonOrchestrator with example_config
    orchestrator = SeasonOrchestrator(season_config=config, store_data=True)
    orchestrator.run_season()

if __name__ == "__main__":
    prof = cProfile.Profile()
    prof.enable()

    main()

    prof.disable()
    prof.dump_stats("prof.stats")

    p = pstats.Stats("prof.stats")
    p.strip_dirs().sort_stats("cumulative").print_stats(40)




# Example configs

In [ ]:
# ===================== Example Toll Configurations =====================
# Uncomment and use any of these in make_season_config(toll=..., bus_user_fee=...)

# 1. Static toll (fixed price)
# toll=TollConfig.static(car=10.0),

# 2. Volume-based piecewise linear (current config above)
# toll=TollConfig(
#     signal=VolumeSignal(),
#     transform=PiecewiseLinearTransform(threshold=100, slope=0.05, base=5.0),
#     update_every_n_steps=60,
#     rounding=0.25,
# ),

# 3. Flow-based piecewise linear (rolling average arrival rate)
# toll=TollConfig(
#     signal=FlowSignal(window_steps=300),  # 5-minute rolling window
#     transform=PiecewiseLinearTransform(threshold=1.0, slope=10.0, base=2.0),
#     update_every_n_steps=60,
#     rounding=0.25,
#     cap=25.0,  # max toll $25
# ),

# 4. Volume-based step toll (binary: $0 or $10)
# toll=TollConfig(
#     signal=VolumeSignal(),
#     transform=StepTransform(threshold=100, toll=10.0),
#     update_every_n_steps=60,
# ),

# 5. Volume-based PI controller (feedback-driven)
# toll=TollConfig(
#     signal=VolumeSignal(),
#     transform=PITransform(target=300, kp=0.5, ki=0.05, toll_min=0, toll_max=50),
#     update_every_n_steps=60,
#     rounding=0.10,
# ),